# Late Chunking: Mechanics Explained

**Late Chunking** is an embedding technique that preserves cross-chunk context by running the full document through the transformer first, then slicing the token embeddings — rather than embedding each chunk in isolation.

## The Core Idea

| Approach | Process |
|---|---|
| **Early Chunking** | Split text → embed each chunk separately → lose cross-chunk context |
| **Late Chunking** | Embed full text → split the resulting token embeddings → context is preserved |

This notebook walks through both approaches on a single sentence so you can see exactly what changes.

## Step 1 — Imports

In [1]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

## Step 2 — Load Model

We use `BAAI/bge-small-en-v1.5`, a lightweight but capable sentence embedding model.

In [2]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

print(f"Loaded model: {MODEL_NAME}")
print(f"Hidden size (embedding dim): {model.config.hidden_size}")

Loaded model: BAAI/bge-small-en-v1.5
Hidden size (embedding dim): 384


## Step 3 — Define a Small Sentence and Its Chunks

We split the sentence into two chunks:
- **Chunk A**: mentions the iPhone's titanium frame
- **Chunk B**: says "it" improves durability — where "it" refers to the titanium frame

This pronoun reference is exactly the kind of cross-chunk context that early chunking loses.

In [3]:
sentence = "The iPhone has a titanium frame and it improves durability"

chunk_a = "The iPhone has a titanium frame"
chunk_b = "it improves durability"

query = "What improves durability?"

print(f"Full sentence : {sentence}")
print(f"Chunk A       : {chunk_a}")
print(f"Chunk B       : {chunk_b}")
print(f"Query         : {query}")

Full sentence : The iPhone has a titanium frame and it improves durability
Chunk A       : The iPhone has a titanium frame
Chunk B       : it improves durability
Query         : What improves durability?


## Step 4 — Embed Chunks Alone (Early Chunking)

Each chunk is tokenized and embedded **independently**. The model never sees the other chunk, so "it" in Chunk B has no referent.

In [4]:
def mean_pool(token_embeddings: torch.Tensor, attention_mask: torch.Tensor) -> np.ndarray:
    """Mean-pool token embeddings, ignoring padding tokens."""
    mask = attention_mask.unsqueeze(-1).float()
    summed = (token_embeddings * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return (summed / counts).detach().numpy()


def embed_text(text: str) -> np.ndarray:
    """Tokenize a string and return its mean-pooled embedding."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return mean_pool(outputs.last_hidden_state, inputs["attention_mask"])


early_a = embed_text(chunk_a)   # shape: (1, hidden_size)
early_b = embed_text(chunk_b)   # shape: (1, hidden_size)

print(f"Early embedding shape: {early_a.shape}")

Early embedding shape: (1, 384)


## Step 5 — Print Early Chunking Vectors

In [5]:
np.set_printoptions(precision=4, suppress=True, linewidth=120)

print("Early Chunk A vector (first 16 dims):")
print(early_a[0, :16])

print("\nEarly Chunk B vector (first 16 dims):")
print(early_b[0, :16])

Early Chunk A vector (first 16 dims):
[-0.3405  0.2231  0.0766 -0.2805 -0.3722 -0.3831  0.7281  0.0641  0.1125  0.3869 -0.0839 -0.1574  0.1671  0.4173
  0.2492  0.0442]

Early Chunk B vector (first 16 dims):
[-0.7501  0.0094  0.7388 -0.1031  0.3071  0.1651  0.0675  0.7076 -0.2029 -0.0394 -0.2693 -0.2827  0.683   0.3988
  0.0237 -0.1124]


## Step 6 — Embed the Full Sentence

Now we pass the **entire sentence** through the model in one go. Every token attends to every other token via self-attention, so "it" can attend to "titanium frame".

In [6]:
full_inputs = tokenizer(sentence, return_tensors="pt", truncation=True)

with torch.no_grad():
    full_outputs = model(**full_inputs)

# Shape: (1, num_tokens, hidden_size)
full_token_embeddings = full_outputs.last_hidden_state

tokens = tokenizer.convert_ids_to_tokens(full_inputs["input_ids"][0])
print(f"Tokens in full sentence ({len(tokens)} total):")
for i, tok in enumerate(tokens):
    print(f"  [{i:2d}] {tok}")

Tokens in full sentence (14 total):
  [ 0] [CLS]
  [ 1] the
  [ 2] iphone
  [ 3] has
  [ 4] a
  [ 5] titanium
  [ 6] frame
  [ 7] and
  [ 8] it
  [ 9] improves
  [10] du
  [11] ##ra
  [12] ##bility
  [13] [SEP]


## Step 7 — Identify Chunk Token Positions Inside the Full Sentence

We need to know *which token indices* in the full sequence correspond to each chunk. We do this by tokenizing each chunk separately and then finding its token span inside the full token list.

In [7]:
def find_chunk_span(full_tokens: list[str], chunk_text: str) -> tuple[int, int]:
    """
    Returns (start, end) token indices (exclusive end) of chunk_text
    within full_tokens, skipping [CLS] / [SEP] special tokens.
    """
    chunk_ids = tokenizer(chunk_text, add_special_tokens=False)["input_ids"]
    chunk_tokens = tokenizer.convert_ids_to_tokens(chunk_ids)

    # Search for the sub-sequence in full_tokens
    n = len(chunk_tokens)
    for i in range(len(full_tokens) - n + 1):
        if full_tokens[i : i + n] == chunk_tokens:
            return i, i + n

    raise ValueError(f"Chunk tokens not found in full sequence.\n  chunk: {chunk_tokens}\n  full: {full_tokens}")


span_a = find_chunk_span(tokens, chunk_a)
span_b = find_chunk_span(tokens, chunk_b)

print(f"Chunk A '{chunk_a}'")
print(f"  Token span: {span_a}  → {tokens[span_a[0]:span_a[1]]}")
print()
print(f"Chunk B '{chunk_b}'")
print(f"  Token span: {span_b}  → {tokens[span_b[0]:span_b[1]]}")

Chunk A 'The iPhone has a titanium frame'
  Token span: (1, 7)  → ['the', 'iphone', 'has', 'a', 'titanium', 'frame']

Chunk B 'it improves durability'
  Token span: (8, 13)  → ['it', 'improves', 'du', '##ra', '##bility']


## Step 8 — Slice Token Embeddings (Late Chunking)

We take the context-aware token embeddings from the full sentence and **mean-pool only the tokens belonging to each chunk**. These embeddings carry information from the full sentence's attention, unlike early chunking.

In [8]:
def late_chunk_embed(token_embeddings: torch.Tensor, span: tuple[int, int]) -> np.ndarray:
    """Mean-pool a slice of context-aware token embeddings."""
    start, end = span
    chunk_vecs = token_embeddings[0, start:end, :]  # (chunk_len, hidden_size)
    return chunk_vecs.mean(dim=0, keepdim=True).detach().numpy()  # (1, hidden_size)


late_a = late_chunk_embed(full_token_embeddings, span_a)
late_b = late_chunk_embed(full_token_embeddings, span_b)

print("Late Chunk A vector (first 16 dims):")
print(late_a[0, :16])

print("\nLate Chunk B vector (first 16 dims):")
print(late_b[0, :16])

Late Chunk A vector (first 16 dims):
[-0.6424  0.1701  0.3392 -0.1428 -0.317  -0.1014  0.451   0.265   0.0299  0.2895 -0.1027  0.111   0.3701  0.4802
  0.2165 -0.2271]

Late Chunk B vector (first 16 dims):
[-0.7976 -0.0269  0.4264 -0.0442 -0.1614 -0.1498  0.2611  0.411  -0.2404  0.3639 -0.1584 -0.0257  0.5113  0.5705
 -0.0032 -0.1658]


## Step 9 — Compare Early vs Late Chunking with a Query

Query: **"What improves durability?"**

Late chunking does not raise every chunk's similarity. It changes what each chunk's vector represents, so here is what we should expect:

- **Chunk B** (`"it improves durability"`) almost repeats the query, so early chunking gives it a very high score. That is a false positive: read alone, the chunk never says what "it" is.
- In late chunking, Chunk B's tokens have attended to the whole sentence, including `iPhone` and `titanium frame`. Its vector is no longer a pure "improves durability" signal, so its similarity should drop.
- **Chunk A** (`"The iPhone has a titanium frame"`) holds the real answer. Its tokens have now attended to `improves durability`, so its similarity should rise.

The code below measures both, and also shows whether the ranking changes.

In [9]:
query_vec = embed_text(query)  # shape: (1, hidden_size)


def similarity(chunk_vec: np.ndarray) -> float:
    return cosine_similarity(query_vec, chunk_vec)[0, 0]


sim_early_a, sim_early_b = similarity(early_a), similarity(early_b)
sim_late_a, sim_late_b = similarity(late_a), similarity(late_b)

print(f"Query: '{query}'")
print()
print("=" * 55)
print(f"{'Chunk':<30} {'Early':>10} {'Late':>10}")
print("-" * 55)
print(f"{'A: ' + repr(chunk_a):<30} {sim_early_a:>10.4f} {sim_late_a:>10.4f}")
print(f"{'B: ' + repr(chunk_b):<30} {sim_early_b:>10.4f} {sim_late_b:>10.4f}")
print("=" * 55)
print()
print(f"Chunk A changed by {sim_late_a - sim_early_a:+.4f}, Chunk B by {sim_late_b - sim_early_b:+.4f}")
print(f"B's lead over A: {sim_early_b - sim_early_a:.4f} early, {sim_late_b - sim_late_a:.4f} late")
print(f"Top chunk: {'A' if sim_early_a > sim_early_b else 'B'} early, {'A' if sim_late_a > sim_late_b else 'B'} late")

Query: 'What improves durability?'



Chunk                               Early       Late
-------------------------------------------------------
A: 'The iPhone has a titanium frame'     0.4112     0.5372
B: 'it improves durability'        0.8682     0.7063

Chunk A changed by +0.1259, Chunk B by -0.1619
B's lead over A: 0.4570 early, 0.1691 late
Top chunk: B early, B late


**Reading the result:** both chunks moved the expected way. The real answer (A) rose, the false positive (B) fell, and B's lead shrank by about 60%. But B still ranks first, so on this sentence late chunking narrows the gap and does not flip the ranking.

## Step 10 — Try Other Queries

One query can't show the whole picture. These four run against the same two chunks, including questions where Chunk A is the obvious answer.

In [10]:
other_queries = [
    "What improves durability?",
    "What is the iPhone frame made of?",
    "Does titanium make a phone more durable?",
    "Why is the iPhone durable?",
]

print(f"{'Query':<42}{'A early':>9}{'A late':>9}{'B early':>9}{'B late':>9}{'Top early':>11}{'Top late':>10}")
print("-" * 99)
for text in other_queries:
    vec = embed_text(text)
    a_early, b_early = cosine_similarity(vec, early_a)[0, 0], cosine_similarity(vec, early_b)[0, 0]
    a_late, b_late = cosine_similarity(vec, late_a)[0, 0], cosine_similarity(vec, late_b)[0, 0]
    top_early = "A" if a_early > b_early else "B"
    top_late = "A" if a_late > b_late else "B"
    print(f"{text:<42}{a_early:>9.4f}{a_late:>9.4f}{b_early:>9.4f}{b_late:>9.4f}{top_early:>11}{top_late:>10}")

Query                                       A early   A late  B early   B late  Top early  Top late
---------------------------------------------------------------------------------------------------
What improves durability?                    0.4112   0.5372   0.8682   0.7063          B         B
What is the iPhone frame made of?            0.8295   0.8008   0.4715   0.7394          A         A


Does titanium make a phone more durable?     0.6734   0.7741   0.6870   0.8105          B         B
Why is the iPhone durable?                   0.6984   0.7818   0.6802   0.8159          A         B


**Late chunking pulls neighbouring chunks towards their shared context, whichever one holds the answer.** Chunk B jumps from 0.47 to 0.74 on "What is the iPhone frame made of?", even though it says nothing about the frame. On "Why is the iPhone durable?" the top chunk flips from A to B.

So chunks gain context, but they also become more alike. Two chunks of one short sentence is an extreme case, and the effect depends on the document. [`rag06.py`](rag06.py) measures retrieval on a longer document, which is the right way to judge whether late chunking helps.

## Summary

| | Early Chunking | Late Chunking |
|---|---|---|
| **When split happens** | Before the model | After the model |
| **Cross-chunk context** | Lost | Carried in through self-attention |
| **Pronouns** | "it" has no referent | Tokens can attend to the referent |
| **Compute cost** | One forward pass per chunk | One forward pass for the full text |
| **Document length** | Any | Limited by the model's input (512 tokens here) |
| **Best for** | Short, self-contained chunks | Documents with references that cross chunk boundaries |

Late chunking is most useful when chunks contain pronouns, ellipses or entity references that span chunk boundaries. Measure it on your own documents before relying on it, because adding context can also make chunks harder to tell apart.